# 🪐 Cosmopedia v2 Teacher-Digested Pretraining Pipeline
### High-Throughput Sharded Ingestion with Qwen2.5-1.5B-Instruct & Hugging Face Hub Publication
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/Cosmopedia_V2_Teacher_Digestion_Pipeline.ipynb)

---

### Pipeline Architecture & Objectives
This notebook orchestrates the end-to-end ingestion and digestion of the **Cosmopedia v2** dataset (39.13 Million documents across 104 shards from `HuggingFaceTB/smollm-corpus`) into an information-dense pretraining corpus stored on Hugging Face Hub ([AjitKumar-Bio/cosmopedia-v2-digested](https://huggingface.co/datasets/AjitKumar-Bio/cosmopedia-v2-digested)).

1. **Teacher Digestion**: Uses `Qwen/Qwen2.5-1.5B-Instruct` to compute per-token predictive surprisal $\mathcal{L}_{Q, t} = -\log P_Q(x_t \mid x_{<t})$ and predictive entropy $\mathcal{H}_{Q, t}$.
2. **Cross-Tokenizer Offset Projection**: Losslessly projects information-density weights from Qwen\'s 152k vocabulary onto CerebroSpheroid\'s 32k SentencePiece vocabulary via character spans $[c_{\text{start}}, c_{\text{end}})$.
3. **Dynamic Next-Token Loss Weights**: Yields normalized weights $w_t \in [0.2, 4.0]$ ($E[w_t] = 1.0$) to bias the Spheroid network towards high-entropy reasoning and semantic syntax.
4. **Sharded & Resumable Streaming**: Downloads 1 shard at a time (~1.17 GB), digests samples with low RAM overhead, uploads the shard to Hugging Face Hub, and cleans local caches. If restarted, it detects existing shards on Hugging Face and resumes automatically!
5. **Fleet Parallelism**: Allows multiple Colab instances or cloud GPUs to run concurrently via `--worker-id` and `--num-workers`.


---
## Step 1: Environment Setup & Hardware Acceleration
Verify GPU accelerator (T4, V100, A100, L4) and install necessary Hugging Face and PyTorch dependencies.


In [ ]:
import os
import sys
import torch

print("=" * 70)
print("🧠 ENVIRONMENT & ACCELERATOR CHECK")
print("=" * 70)

device_name = "CPU"
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"  ✓ Active GPU       : {device_name} ({vram_gb:.1f} GB VRAM)")
    print(f"  ✓ Bfloat16 Support : {bf16_ok}")
else:
    print("  ⚠️ CUDA not detected. Running on CPU (slower).")

# Install dependencies
!pip install -q --upgrade transformers accelerate datasets huggingface_hub pyarrow sentencepiece
print("  ✓ Dependencies successfully installed.")


---
## Step 2: Clone or Update CerebroSpheroid V5 Repository
Pulls the latest code from GitHub into Google Colab.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git"
REPO_DIR = "/content/cerebro_spheroid_v5"

if not os.path.exists(REPO_DIR):
    print(f"🔄 Cloning CerebroSpheroid V5 repository from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Pulling latest updates into {REPO_DIR}...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✓ Active workspace: {os.getcwd()}")


---
## Step 3: Hugging Face Hub Authentication
Log in with your Hugging Face write token to allow automated shard uploads to your repository.


In [ ]:
import os
from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Successfully authenticated via HF_TOKEN.")
else:
    print("🔑 Please enter your Hugging Face Access Token (write permissions required):")
    login()

api = HfApi()
user_info = api.whoami()
print(f"✓ Authenticated as: {user_info["name"]}")


---
## Step 4: Configure Digestion Pipeline Run
Configure the target repository, shard indices, and worker distribution.

- **`TARGET_REPO_ID`**: Target dataset repository on Hugging Face (default: `AjitKumar-Bio/cosmopedia-v2-digested`).
- **`SHARD_START` / `SHARD_END`**: Range of shards to digest (0 to 104).
- **`WORKER_ID` / `NUM_WORKERS`**: Distribute work across multiple Colab instances (e.g., Worker 0 of 4 processes shards 0, 4, 8, ...).
- **`MAX_SAMPLES`**: Set to `None` for full shards (~376k docs each), or an integer (e.g., `1000`) for quick batches.


In [ ]:
# --------------------------------------------------------------------------
# Pipeline Execution Parameters
# --------------------------------------------------------------------------
TARGET_REPO_ID = "AjitKumar-Bio/cosmopedia-v2-digested"
SOURCE_REPO    = "HuggingFaceTB/smollm-corpus"
TOTAL_SHARDS   = 104

# Shard Range / Selection
SHARD_START    = 0
SHARD_END      = 5         # Set to 104 to process all shards through the end
SHARD_INDICES  = ""        # Optional comma-separated specific shards, e.g. "0,1,2"

# Distributed Fleet Parameters
WORKER_ID      = 0         # 0, 1, 2, ...
NUM_WORKERS    = 1         # Set to >1 if running multiple Colabs in parallel

# Digestion Parameters
MAX_SAMPLES    = None      # Set to None for full shards (~376k docs), or e.g. 5000 for smoke testing
MAX_SEQ_LEN    = 1024      # Cosmopedia v2 documents average ~800 tokens
TEACHER_SCALE  = 1.5       # Dynamic loss alpha scaling
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
PUSH_TO_HUB    = True
SKIP_EXISTING  = True      # Auto-skips shards already uploaded to Hugging Face
# --------------------------------------------------------------------------

print(f"🚀 Configured Pipeline:")
print(f"  Target Hub Repo : {TARGET_REPO_ID}")
print(f"  Shard Range     : [{SHARD_START} -> {SHARD_END}) of {TOTAL_SHARDS}")
print(f"  Worker          : {WORKER_ID + 1} of {NUM_WORKERS}")
print(f"  Max Samples     : {"Full Shard" if MAX_SAMPLES is None else f"{MAX_SAMPLES:,} per shard"}")


---
## Step 5: Launch Sharded Cosmopedia v2 Teacher Digestion
Executes the digestion pipeline. Completed shards are immediately streamed to Hugging Face Hub, and local caches are pruned to prevent disk exhaustion.


In [ ]:
cmd = [
    "python3", "scripts/digest_cosmopedia_v2.py",
    "--repo-id", TARGET_REPO_ID,
    "--source-repo", SOURCE_REPO,
    "--total-shards", str(TOTAL_SHARDS),
    "--shard-start", str(SHARD_START),
    "--shard-end", str(SHARD_END),
    "--worker-id", str(WORKER_ID),
    "--num-workers", str(NUM_WORKERS),
    "--max-length", str(MAX_SEQ_LEN),
    "--teacher-scale", str(TEACHER_SCALE),
    "--device", DEVICE,
]

if SHARD_INDICES.strip():
    cmd.extend(["--shard-indices", SHARD_INDICES.strip()])

if MAX_SAMPLES is not None:
    cmd.extend(["--max-samples-per-shard", str(MAX_SAMPLES)])

if not PUSH_TO_HUB:
    cmd.append("--no-push")

if not SKIP_EXISTING:
    cmd.append("--no-skip")

print("Executing command:\n", " ".join(cmd), "\n")
!{" ".join(cmd)}


---
## Step 6: Multi-Worker Distributed Fleet Deployment Guide

To digest all 104 shards rapidly, you can run multiple Colab notebooks concurrently across free or Pro GPU instances:

| Colab Session | `WORKER_ID` | `NUM_WORKERS` | Shards Processed |
|---|---|---|---|
| **Colab 1** | `0` | `4` | 0, 4, 8, 12, ..., 100 |
| **Colab 2** | `1` | `4` | 1, 5, 9, 13, ..., 101 |
| **Colab 3** | `2` | `4` | 2, 6, 10, 14, ..., 102 |
| **Colab 4** | `3` | `4` | 3, 7, 11, 15, ..., 103 |

Because `--skip-existing` is enabled by default, each worker queries Hugging Face Hub before starting a shard and skips any shard uploaded by another worker or a previous run. There is zero risk of collisions.


---
## Step 7: Stream Digested Cosmopedia v2 Directly into CerebroSpheroid Pretraining
Test streaming directly from the uploaded Hugging Face dataset during pretraining with **zero runtime Qwen inference overhead** and real-time dynamic token weighting.


In [ ]:
# Verify pretraining stream directly from the Hugging Face dataset
!python3 scripts/train_v5_compact_growth_pretrain.py \
    --train-data {TARGET_REPO_ID} \
    --digest-mode none \
    --total-steps 10 \
    --device {DEVICE}
